# Kuvageneraattorien sovellusten rakentaminen (OpenAI)

Suurissa kielimalleissa on enemmänkin kuin pelkkä tekstintuotanto. Voit myös luoda kuvia tekstikuvauksista. Kuvien käyttö modaliteettina on hyödyllistä MedTechissä, arkkitehtuurissa, matkailussa, pelikehityksessä, markkinoinnissa ja monessa muussa. Tässä oppitunnissa työskentelemme nykyisten **GPT Image** -mallien kanssa OpenAI-alustalla.

## Oppimistavoitteet

Tämän oppitunnin lopussa osaat:

- Selittää, mitä kuvagenerointi on ja missä sitä voi hyödyntää.
- Ymmärtää `gpt-image` -malliperheen sekä miten se eroaa perinteisistä DALL·E-malleista.
- Rakentaa kuvageneraattorisovelluksen ja muokata kuvia.

## Mitä on kuvagenerointi?

Kuvageneraattorimallit luovat kuvia tekstikehotteen perusteella. Nykyaikaiset mallit, kuten `gpt-image`, oppivat tekstin ja kuvien välisen yhteyden koulutuksessa, ja muuttavat sitten iteratiivisesti satunnaisen kohinan kuvaksi, joka vastaa kehotettasi.

Kaksi hyvin tunnettua kuvamalliperhettä ovat:

- **`gpt-image` (OpenAI)** — tämä oppitunti käsittelee tämän nykyisen sukupolven mallia. Se tukee tekstistä kuvaan -generointia ja kuvien muokkausta (maalaus maskin avulla).
- **Midjourney** — suosittu kolmannen osapuolen malli, jolla on oma palvelunsa ja Discord-pohjainen toimintamalli.

> Vanhemmat OpenAI:n kuvamallit — **DALL·E 2** ja **DALL·E 3** — ovat perinteisiä. DALL·E 3 ei ole enää saatavilla uusille käyttöönotolle, ja ominaisuudet kuten `create_variation` olivat käytettävissä vain DALL·E 2:ssa. Käytä `gpt-image` -malleja uusissa sovelluksissa.

> **Tärkeää:** `gpt-image` -mallit palauttavat generoimansa kuvan **base64**-muodossa (`b64_json`), eivät URL-osoitteena. Koodisi purkaa base64-merkkijonon tavuiksi ja tallentaa sen — ladattavaa kuva-URLia ei siis ole.


## Ensimmäisen kuvanluontisovelluksesi rakentaminen

Mitä siis tarvitaan kuvanluontisovelluksen rakentamiseen? Tarvitset seuraavat kirjastot:

- **python-dotenv**, tätä kirjastoa suositellaan voimakkaasti salaisuuksien tallentamiseen *.env* -tiedostoon erilleen koodista.
- **openai**, tätä kirjastoa käytät kommunikoidaksesi OpenAI API:n kanssa.
- **pillow**, kuvien käsittelyyn Pythonissa.


1. Luo *.env* -tiedosto, jonka sisältö on seuraava:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Kerää yllä olevat kirjastot tiedostoon nimeltä *requirements.txt* seuraavasti:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Luo sitten virtuaaliympäristö ja asenna kirjastot:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Windowsille käytä seuraavia komentoja virtuaaliympäristön luomiseen ja aktivointiin:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Lisää seuraava koodi tiedostoon nimeltä *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Luo OpenAI-objekti (lukee OPENAI_API_KEY .env-tiedostostasi)
    client = openai.OpenAI()


    try:
        # Luo kuva käyttämällä kuvanluonti-APIa
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Syötä tähän tekstisi
            size='1024x1024',
            n=1
        )
        # Aseta hakemisto tallennetulle kuvalle
        image_dir = os.path.join(os.curdir, 'images')

        # Jos hakemistoa ei ole, luo se
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Alusta kuvan polku (huomaa, että tiedostotyyppi tulee olla png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image-mallit palauttavat kuvan base64-muodossa (b64_json), ei URL-osoitteena
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Näytä kuva oletuskuvankatseluohjelmassa
        image = Image.open(image_path)
        image.show()

    # käsittele poikkeukset
    except openai.BadRequestError as err:
        print(err)

    ```

Selitetään tämä koodi:

- Ensin tuomme kirjastot, joita tarvitsemme, mukaan lukien OpenAI-kirjasto, dotenv-kirjasto, base64-moduuli ja Pillow-kirjasto.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Sen jälkeen luomme asiakkaan, joka lukee API-avaimen tiedostostasi ``.env``.

    ```python
    # Luo OpenAI-olio
    client = openai.OpenAI()
    ```

- Seuraavaksi generoimme kuvan:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Syötä kehotetekstisi tähän
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` -mallit palauttavat kuvan **base64**-merkkijonona `data[0].b64_json`. Dekoodamme sen tavuiksi ja kirjoitamme tiedostoon — latausosoitetta ei ole.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Lopuksi avaamme kuvan ja käytämme tavallista kuvan katseluohjelmaa näyttääksesi sen:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Lisätietoja kuvan generoinnista

Tarkastellaan `images.generate` -funktion parametreja:

- **model**, on käytettävä kuvan malli (esimerkiksi `gpt-image-1`).
- **prompt**, on tekstiprompti, jota käytetään kuvan luomiseen. Tässä se on "Jänis hevosella, pitelee tikkaria sumuisella niityllä, jossa kasvaa narsisseja".
- **size**, on luodun kuvan koko (esimerkiksi `1024x1024`, `1536x1024`, `1024x1536` tai `"auto"`).
- **n**, on luotujen kuvien määrä. Tässä luodaan yksi.

> Kuvamallit eivät käytä `temperature`-parametria — se on tekstin generoinnin ohjaus. Vaihtelun saamiseksi kutsu API uudelleen; vaihtelun vähentämiseksi tee promptistasi tarkempi.

## Kuvageneroinnin lisäominaisuudet

Olet nähnyt, miten kuvageneraattori toimii muutamalla Python-rivillä. `gpt-image` -mallit voivat myös **muokata** olemassa olevaa kuvaa. Antamalla kuvan, valinnaisen **maskin** (joka merkitsee muutettavan alueen) ja promptin, voit muuttaa osaa kuvasta — esimerkiksi lisätä hatun jäniksellemme.

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# muokkaukset palautetaan myös base64-muodossa
edited_image = base64.b64decode(response.data[0].b64_json)
```

Peruskuva sisältää vain jäniksen; lopullisessa kuvassa jäniksellä on hattu.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
